<a href="https://colab.research.google.com/github/16howarthm/pgh-road-ai/blob/main/category_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pittsburgh street-closure category analysis
Reproducible experiment with six balanced categories, fixed prompt-development and held-out splits, and exact 0/1/several-example prompt conditions.

In [4]:
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version as package_version
import ast
import base64
import importlib
import os
import json
import re
import shutil
import subprocess
import sys
import tempfile

REPOSITORY_URL = "https://github.com/16howarthm/pgh-road-ai.git"
COLAB_REPOSITORY_DIR = Path("/content/pgh-road-ai")
CHECKPOINT_RELATIVE_PATH = Path("category_results/classification_results.csv")
REQUIRED_REPOSITORY_FILES = [
    "road_ai.py",
    "pgh data.csv",
    "category_splits/prompt_development.csv",
    "category_splits/heldout_test.csv",
]
REQUIRED_ROAD_AI_SYMBOLS = {
    "DEFAULT_CATEGORIES",
    "LLAMA_SCOUT_MODEL",
    "balanced_sample",
    "completed_result_keys",
    "load_fixed_splits",
    "load_results_checkpoint",
    "make_examples",
    "remove_model_routing_failures",
    "run_experiment",
    "save_fixed_splits",
    "save_results_checkpoint",
    "score_results",
    "split_prompt_test",
    "validate_splits",
}

def get_secret(name):
    value = os.getenv(name)
    if value:
        return value
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

def authenticated_git_environment(token):
    if not token:
        raise RuntimeError(
            "GITHUB_TOKEN is unavailable. Add it to Colab Secrets, grant notebook access, "
            "and rerun this cell."
        )
    authorization = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    environment = os.environ.copy()
    environment.pop("GITHUB_TOKEN", None)
    environment.pop("OPENROUTER_API_KEY", None)
    environment.pop("openrouter_api_key", None)
    environment.update({
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {authorization}",
        "GIT_TERMINAL_PROMPT": "0",
    })
    return environment

def git_output(target, *arguments, environment=None):
    return subprocess.run(
        ["git", "-C", str(target), *arguments],
        check=True, env=environment, capture_output=True, text=True,
    ).stdout.strip()

def normalized_repository_url(value):
    value = str(value).strip().rstrip("/")
    if value.startswith("git@github.com:"):
        value = "https://github.com/" + value.split(":", 1)[1]
    return value.removesuffix(".git")

def is_valid_git_repository(path):
    path = Path(path)
    if not path.is_dir():
        return False
    try:
        root = Path(git_output(path, "rev-parse", "--show-toplevel")).resolve()
    except (OSError, subprocess.CalledProcessError):
        return False
    return root == path.resolve()

def repository_origin_matches(path, repository_url=REPOSITORY_URL):
    try:
        origin = git_output(path, "remote", "get-url", "origin")
    except (OSError, subprocess.CalledProcessError):
        return False
    return normalized_repository_url(origin) == normalized_repository_url(repository_url)

def defined_road_ai_symbols(path):
    source_path = Path(path) / "road_ai.py"
    if not source_path.is_file():
        return set()
    try:
        tree = ast.parse(source_path.read_text(encoding="utf-8"))
    except (OSError, SyntaxError):
        return set()
    symbols = set()
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            symbols.add(node.name)
        elif isinstance(node, ast.Assign):
            symbols.update(
                target.id for target in node.targets if isinstance(target, ast.Name)
            )
        elif isinstance(node, ast.AnnAssign) and isinstance(node.target, ast.Name):
            symbols.add(node.target.id)
    return symbols

def verify_experiment_repository(path, expected_commit=None, repository_url=REPOSITORY_URL):
    path = Path(path)
    if not is_valid_git_repository(path):
        raise RuntimeError(f"Fresh clone at {path} is not a valid Git repository.")
    if not repository_origin_matches(path, repository_url):
        raise RuntimeError(f"Fresh clone at {path} does not point to the expected repository.")
    missing_files = [name for name in REQUIRED_REPOSITORY_FILES if not (path / name).is_file()]
    if missing_files:
        raise RuntimeError(f"Fresh clone is missing required files: {missing_files}")
    missing_symbols = sorted(REQUIRED_ROAD_AI_SYMBOLS - defined_road_ai_symbols(path))
    if missing_symbols:
        raise RuntimeError(
            "Fresh clone road_ai.py is missing required experiment symbols: "
            + ", ".join(missing_symbols)
        )
    commit = git_output(path, "rev-parse", "HEAD")
    if expected_commit and commit != expected_commit:
        raise RuntimeError(
            f"Fresh clone commit {commit} does not match remote main {expected_commit}."
        )
    return commit

def clone_private_repository(target, repository_url=REPOSITORY_URL, token=None):
    token = token or get_secret("GITHUB_TOKEN")
    environment = authenticated_git_environment(token)
    try:
        subprocess.run(
            [
                "git", "clone", "--quiet", "--depth", "1",
                "--branch", "main", "--single-branch", repository_url, str(target),
            ],
            check=True, env=environment,
        )
    finally:
        environment["GIT_CONFIG_VALUE_0"] = ""
        token = None

def fetch_remote_main(path, token):
    environment = authenticated_git_environment(token)
    try:
        subprocess.run(
            ["git", "-C", str(path), "fetch", "--quiet", "--depth", "1", "origin", "main"],
            check=True, env=environment,
        )
        return git_output(path, "rev-parse", "FETCH_HEAD")
    finally:
        environment["GIT_CONFIG_VALUE_0"] = ""

def remove_path(path):
    path = Path(path)
    if path.is_dir() and not path.is_symlink():
        shutil.rmtree(path)
    elif path.exists() or path.is_symlink():
        path.unlink()

def working_directory_is_within(path):
    try:
        Path.cwd().resolve().relative_to(Path(path).resolve())
        return True
    except ValueError:
        return False

def refresh_colab_repository(target, token, repository_url=REPOSITORY_URL):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    refresh_directory = Path(tempfile.mkdtemp(prefix=".pgh-road-ai-refresh-", dir=target.parent))
    previous_repository = refresh_directory / "previous-repository"
    checkpoint_backup = refresh_directory / "classification_results.csv"
    checkpoint = target / CHECKPOINT_RELATIVE_PATH
    checkpoint_saved = checkpoint.is_file()
    had_previous = target.exists() or target.is_symlink()
    if checkpoint_saved:
        shutil.copy2(checkpoint, checkpoint_backup)
    if had_previous:
        if working_directory_is_within(target):
            os.chdir(target.parent)
        target.rename(previous_repository)
    try:
        clone_private_repository(target, repository_url=repository_url, token=token)
        remote_commit = git_output(target, "rev-parse", "origin/main")
        commit = verify_experiment_repository(
            target, expected_commit=remote_commit, repository_url=repository_url
        )
        if checkpoint_saved:
            restored_checkpoint = target / CHECKPOINT_RELATIVE_PATH
            restored_checkpoint.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(checkpoint_backup, restored_checkpoint)
    except Exception as exc:
        remove_path(target)
        if previous_repository.exists() or previous_repository.is_symlink():
            previous_repository.rename(target)
        remove_path(refresh_directory)
        recovery = " Previous directory was restored." if had_previous else ""
        raise RuntimeError(f"Repository refresh failed: {exc}.{recovery}") from exc
    remove_path(previous_repository)
    remove_path(refresh_directory)
    if checkpoint_saved:
        print("Restored saved classification checkpoint after repository refresh.")
    return target.resolve(), commit

def ensure_colab_repository(target=COLAB_REPOSITORY_DIR, repository_url=REPOSITORY_URL, token=None):
    target = Path(target)
    token = token or get_secret("GITHUB_TOKEN")
    authenticated_git_environment(token)
    needs_refresh = True
    remote_commit = None
    if is_valid_git_repository(target) and repository_origin_matches(target, repository_url):
        try:
            remote_commit = fetch_remote_main(target, token)
            current_commit = git_output(target, "rev-parse", "HEAD")
            missing_files = [
                name for name in REQUIRED_REPOSITORY_FILES if not (target / name).is_file()
            ]
            missing_symbols = REQUIRED_ROAD_AI_SYMBOLS - defined_road_ai_symbols(target)
            needs_refresh = bool(
                current_commit != remote_commit or missing_files or missing_symbols
            )
        except (OSError, subprocess.CalledProcessError) as exc:
            raise RuntimeError("Unable to verify the current remote main commit.") from exc
    if needs_refresh:
        project_root, commit = refresh_colab_repository(
            target, token=token, repository_url=repository_url
        )
    else:
        commit = verify_experiment_repository(
            target, expected_commit=remote_commit, repository_url=repository_url
        )
        project_root = target.resolve()
    token = None
    print("Verified repository commit:", commit)
    return project_root

if Path("/content").exists():
    PROJECT_ROOT = ensure_colab_repository()
elif is_valid_git_repository(Path.cwd()):
    PROJECT_ROOT = Path.cwd().resolve()
    verify_experiment_repository(PROJECT_ROOT)
else:
    raise FileNotFoundError("Run this notebook from the pgh-road-ai repository root.")

os.chdir(PROJECT_ROOT)
required_packages = [
    ("pandas", "2.0"),
    ("scikit-learn", "1.3"),
    ("openai", "1.0"),
]
to_install = []
for package, minimum in required_packages:
    try:
        installed = package_version(package)
        installed_parts = tuple(map(int, re.findall(r"\d+", installed)[:2]))
        minimum_parts = tuple(map(int, minimum.split(".")))
        if installed_parts < minimum_parts:
            to_install.append(f"{package}>={minimum}")
    except PackageNotFoundError:
        to_install.append(f"{package}>={minimum}")
if to_install:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *to_install],
        check=True,
    )

import pandas as pd
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "road_ai" or module_name.startswith("road_ai."):
        del sys.modules[module_name]
sys.path = [
    entry for entry in sys.path
    if not entry or Path(entry).resolve() != PROJECT_ROOT
]
sys.path.insert(0, str(PROJECT_ROOT))
road_ai = importlib.import_module("road_ai")
if Path(road_ai.__file__).resolve() != (PROJECT_ROOT / "road_ai.py").resolve():
    raise RuntimeError(f"Imported road_ai from unexpected location: {road_ai.__file__}")
missing_runtime_symbols = sorted(
    name for name in REQUIRED_ROAD_AI_SYMBOLS if not hasattr(road_ai, name)
)
if missing_runtime_symbols:
    raise RuntimeError(
        "Freshly imported road_ai is missing required experiment symbols: "
        + ", ".join(missing_runtime_symbols)
    )
from road_ai import (
    DEFAULT_CATEGORIES,
    LLAMA_SCOUT_MODEL,
    balanced_sample,
    completed_result_keys,
    load_fixed_splits,
    load_results_checkpoint,
    make_examples,
    remove_model_routing_failures,
    run_experiment,
    save_fixed_splits,
    save_results_checkpoint,
    score_results,
    split_prompt_test,
    validate_splits,
)

if not os.getenv("openrouter_api_key"):
    openrouter_key = get_secret("openrouter_api_key")
    if openrouter_key:
        os.environ["openrouter_api_key"] = openrouter_key
    openrouter_key = None

DATA_PATH = PROJECT_ROOT / "pgh data.csv"
SPLIT_DIR = PROJECT_ROOT / "category_splits"
OUTPUT_DIR = PROJECT_ROOT / "category_results"
LABEL_COL = "work_type"
CATEGORIES = DEFAULT_CATEGORIES
MODELS = [LLAMA_SCOUT_MODEL, "openai/gpt-oss-120b"]
PROMPT_CONDITIONS = {"zero": 0, "one": 1, "several": 6}
RANDOM_STATE = 42

print("Project root:", PROJECT_ROOT)
print("OpenRouter key available:", bool(os.getenv("openrouter_api_key")))

Project root: /content/pgh-road-ai
OpenRouter key available: True


## Load or create the fixed splits
The first run de-duplicates permits, reports and removes conflicting-label description groups, samples 50 unique descriptions per category, and saves fixed 120/180 splits. Later runs load those same records.

In [2]:
prompt_path = SPLIT_DIR / "prompt_development.csv"
test_path = SPLIT_DIR / "heldout_test.csv"
audit_path = SPLIT_DIR / "split_audit.json"

if prompt_path.exists() and test_path.exists():
    prompt_dev, test = load_fixed_splits(LABEL_COL, SPLIT_DIR)
    audit = json.loads(audit_path.read_text()) if audit_path.exists() else {}
else:
    df = pd.read_csv(DATA_PATH)
    sample, audit = balanced_sample(
        df, LABEL_COL, CATEGORIES, n_per_category=50,
        random_state=RANDOM_STATE, return_audit=True,
    )
    prompt_dev, test = split_prompt_test(
        sample, LABEL_COL, prompt_n=120, test_n=180, random_state=RANDOM_STATE
    )
    save_fixed_splits(prompt_dev, test, LABEL_COL, SPLIT_DIR, audit)

validate_splits(prompt_dev, test, LABEL_COL)
print(audit)
print("Prompt-development counts:\n", prompt_dev[LABEL_COL].value_counts().sort_index())
print("Held-out counts:\n", test[LABEL_COL].value_counts().sort_index())

{'ambiguous_description_groups': 28, 'ambiguous_records_excluded': 505, 'eligible_permits_before_ambiguity_filter': 12746, 'random_state': 42, 'records_per_category': 50, 'sample_size': 300}
Prompt-development counts:
 work_type
BARRICADE              20
CRANE                  20
DEMOLITION DUMPSTER    20
MACHINERY              20
MATERIALS              20
SCAFFOLD               20
Name: count, dtype: int64
Held-out counts:
 work_type
BARRICADE              30
CRANE                  30
DEMOLITION DUMPSTER    30
MACHINERY              30
MATERIALS              30
SCAFFOLD               30
Name: count, dtype: int64


## Run every model and prompt condition
Each run uses the same saved 180-record held-out set. `one` contains exactly one labeled example total; `several` contains six total (one per category). Before running this cell in Colab, add `OPENROUTER_API_KEY` under the notebook's **Secrets** panel and grant notebook access.

In [6]:
if not os.getenv("openrouter_api_key"):
    raise RuntimeError(
        "openrouter_api_key is unavailable. Add it to Colab Secrets, grant notebook access, "
        "and rerun the setup cell."
    )

# Ensure the uppercase key is also set for the library functions
os.environ["OPENROUTER_API_KEY"] = os.getenv("openrouter_api_key")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = OUTPUT_DIR / "classification_results.csv"
results = load_results_checkpoint(
    RESULTS_PATH, MODELS, PROMPT_CONDITIONS, test.permit_id
)
results, routing_failures_removed = remove_model_routing_failures(
    results, LLAMA_SCOUT_MODEL
)
if routing_failures_removed:
    save_results_checkpoint(results, RESULTS_PATH)
    print(
        f"Queued {routing_failures_removed} failed Llama Scout routing requests for retry; "
        "all completed GPT-OSS and successful Llama rows were retained."
    )
completed_keys = completed_result_keys(results)
checkpoint_rows = results.to_dict("records") if not results.empty else []
total_requests = len(MODELS) * len(PROMPT_CONDITIONS) * len(test)

print(f"Resuming with {len(completed_keys)}/{total_requests} requests already saved.")
for model in MODELS:
    for condition, n_examples in PROMPT_CONDITIONS.items():
        examples = make_examples(
            prompt_dev, LABEL_COL, n_examples=n_examples, random_state=RANDOM_STATE
        )
        assert len(examples) == n_examples
        pending_mask = [
            (model, condition, str(permit_id)) not in completed_keys
            for permit_id in test.permit_id
        ]
        pending = test.loc[pending_mask].copy()

        def persist_result(row):
            completed = dict(row)
            completed.update({
                "model": model,
                "prompt_condition": condition,
                "n_examples": n_examples,
            })
            checkpoint_rows.append(completed)
            save_results_checkpoint(pd.DataFrame(checkpoint_rows), RESULTS_PATH)
            completed_keys.add((model, condition, str(completed["permit_id"])))

        if not pending.empty:
            run_experiment(
                pending, LABEL_COL, model=model, categories=CATEGORIES,
                examples=examples, on_result=persist_result,
            )

results = load_results_checkpoint(RESULTS_PATH, MODELS, PROMPT_CONDITIONS, test.permit_id)
if len(results) != total_requests:
    raise RuntimeError(f"Expected {total_requests} saved results; found {len(results)}.")
print("Raw results saved to:", RESULTS_PATH)
results.head()

Raw results saved to: /content/pgh-road-ai/category_results/classification_results.csv


/tmp/ipykernel_1364/4168753241.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat(all_results, ignore_index=True)


,permit_id,description,actual,predicted,confidence,reason,prompt_tokens,completion_tokens,error,model,prompt_condition,n_examples
0,DOMI-CS-2020-01492,ERECT SCAFFOLDING FOR BRICK POINTING.,SCAFFOLD,__FAILED__,None,None,NaN,NaN,Error code: 404 - {'error': {'message': 'No en...,meta-llama/llama-4-scout,zero,0
1,DOMI-CS-2020-01589,FACADE INSPECTIONS USING MAN LIFT,MACHINERY,__FAILED__,None,None,NaN,NaN,Error code: 404 - {'error': {'message': 'No en...,meta-llama/llama-4-scout,zero,0
2,DOMI-CS-2020-02405,CRANE LIFT AT CMU POSNER HALL - ON TECH ST. - ...,CRANE,__FAILED__,None,None,NaN,NaN,Error code: 404 - {'error': {'message': 'No en...,meta-llama/llama-4-scout,zero,0
3,DOMI-CS-2020-03352,SCAFFOLD FOR MASONRY REPOINTING,SCAFFOLD,__FAILED__,None,None,NaN,NaN,Error code: 404 - {'error': {'message': 'No en...,meta-llama/llama-4-scout,zero,0
4,DOMI-CS-2020-03505,"DP-2019-03308DEMOLITION OF 209 FIRST AVENUE, P...",BARRICADE,__FAILED__,None,None,NaN,NaN,Error code: 404 - {'error': {'message': 'No en...,meta-llama/llama-4-scout,zero,0


## Metrics and confusion matrices
Failures and invalid responses remain in the 180-row denominator and appear in the `__FAILED__` confusion-matrix column.

In [7]:
summary_rows = []
matrix_rows = []
for (model, condition), group in results.groupby(["model", "prompt_condition"], sort=False):
    scores = score_results(group, CATEGORIES)
    assert scores["n_scored"] == 180
    summary_rows.append({
        "model": model,
        "prompt_condition": condition,
        "n_examples": int(group.n_examples.iloc[0]),
        "accuracy": scores["accuracy"],
        "macro_f1": scores["macro_f1"],
        "failed_predictions": scores["n_failures"],
        "prompt_tokens": group.prompt_tokens.sum(),
        "completion_tokens": group.completion_tokens.sum(),
    })
    matrix = scores["confusion_matrix"]
    display(model, condition, matrix)
    for actual, row in matrix.iterrows():
        for predicted, count in row.items():
            matrix_rows.append({
                "model": model, "prompt_condition": condition,
                "actual": actual, "predicted": predicted, "count": int(count),
            })

summary = pd.DataFrame(summary_rows)
confusion_matrices = pd.DataFrame(matrix_rows)
summary.to_csv(OUTPUT_DIR / "classification_summary.csv", index=False)
confusion_matrices.to_csv(OUTPUT_DIR / "confusion_matrices.csv", index=False)
print("All outputs saved under:", OUTPUT_DIR)
display(summary)

'meta-llama/llama-4-scout'

'zero'

,MACHINERY,DEMOLITION DUMPSTER,CRANE,BARRICADE,MATERIALS,SCAFFOLD,__FAILED__
MACHINERY,0,0,0,0,0,0,30
DEMOLITION DUMPSTER,0,0,0,0,0,0,30
CRANE,0,0,0,0,0,0,30
BARRICADE,0,0,0,0,0,0,30
MATERIALS,0,0,0,0,0,0,30
SCAFFOLD,0,0,0,0,0,0,30
__FAILED__,0,0,0,0,0,0,0


'meta-llama/llama-4-scout'

'one'

...

'openai/gpt-oss-120b'

'several'

,MACHINERY,DEMOLITION DUMPSTER,CRANE,BARRICADE,MATERIALS,SCAFFOLD,__FAILED__
MACHINERY,21,0,0,3,6,0,0
DEMOLITION DUMPSTER,0,22,0,0,4,1,3
CRANE,4,0,21,0,2,3,0
BARRICADE,2,6,0,16,5,1,0
MATERIALS,1,0,0,1,26,2,0
SCAFFOLD,0,0,0,0,2,28,0
__FAILED__,0,0,0,0,0,0,0


All outputs saved under: /content/pgh-road-ai/category_results


,model,prompt_condition,n_examples,accuracy,macro_f1,failed_predictions,prompt_tokens,completion_tokens
0,meta-llama/llama-4-scout,zero,0,0.000000,0.000000,180,0.0,0.0
1,meta-llama/llama-4-scout,one,1,0.000000,0.000000,180,0.0,0.0
2,meta-llama/llama-4-scout,several,6,0.000000,0.000000,180,0.0,0.0
3,openai/gpt-oss-120b,zero,0,0.761111,0.765012,2,39346.0,37170.0
4,openai/gpt-oss-120b,one,1,0.738889,0.746961,5,41725.0,33556.0
5,openai/gpt-oss-120b,several,6,0.744444,0.750193,3,67011.0,39476.0


## Error review

In [8]:
errors = results[results.actual.ne(results.predicted)].sort_values(
    ["model", "prompt_condition", "confidence"], na_position="first"
)
display(errors[[
    "model", "prompt_condition", "permit_id", "description",
    "actual", "predicted", "confidence", "reason", "error",
]].head(50))

,model,prompt_condition,permit_id,description,actual,predicted,confidence,reason,error
180,meta-llama/llama-4-scout,one,DOMI-CS-2020-01492,ERECT SCAFFOLDING FOR BRICK POINTING.,SCAFFOLD,__FAILED__,None,None,Error code: 404 - {'error': {'message': 'No en...
181,meta-llama/llama-4-scout,one,DOMI-CS-2020-01589,FACADE INSPECTIONS USING MAN LIFT,MACHINERY,__FAILED__,None,None,Error code: 404 - {'error': {'message': 'No en...
182,meta-llama/llama-4-scout,one,DOMI-CS-2020-02405,CRANE LIFT AT CMU POSNER HALL - ON TECH ST. - ...,CRANE,__FAILED__,None,None,Error code: 404 - {'error': {'message': 'No en...
183,meta-llama/llama-4-scout,one,DOMI-CS-2020-03352,SCAFFOLD FOR MASONRY REPOINTING,SCAFFOLD,__FAILED__,None,None,Error code: 404 - {'error': {'message': 'No en...
184,meta-llama/llama-4-scout,one,DOMI-CS-2020-03505,"DP-2019-03308DEMOLITION OF 209 FIRST AVENUE, P...",BARRICADE,__FAILED__,None,None,Error code: 404 - {'error': {'message': 'No en...
185,meta-llama/llama-4-scout,one,DOMI-CS-2020-03630,OVERHEAD SIDEWALK PROTECTION SCAFFOLDING WILL ...,SCAFFOLD,__FAILED__,None,None,Error code: 404 - {'error': {'message': 'No en...
186,meta-llama/llama-4-scout,one,DOMI-CS-2020-03667,DEMOLISH AND REPLACE PLANTED MEDIAN ON FEDERAL...,BARRICADE,__FAILED__,None,None,Error code: 404 - {'error': {'message': 'No en...
187,meta-llama/llama-4-scout,one,DOMI-CS-2020-04621,KITCHEN REMODELING - DUMPSTER COMING TODAY AUG...,DEMOLITION DUMPSTER,__FAILED__,None,None,Error code: 404 - {'error': {'message': 'No en...
188,meta-llama/llama-4-scout,one,DOMI-CS-2020-06434,DEMOLITION DUMPSTER NEEDED TO REMOVE ROOF FROM...,DEMOLITION DUMPSTER,__FAILED__,None,None,Error code: 404 - {'error': {'message': 'No en...
189,meta-llama/llama-4-scout,one,DOMI-CS-2020-06512,BUIDING SCAFFOLD ON THE SIDEWALK TO BE ABLE TO...,SCAFFOLD,__FAILED__,None,None,Error code: 404 - {'error': {'message': 'No en...
